In [20]:
import os
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {"User-Agent": "Mozilla/5.0"}

#image_dir = "../data/images"
image_dir = "../data/images/testing"
os.makedirs(image_dir, exist_ok=True)

def scrape_product(product_link, image_dir="../data/images"):
    row = {}
    os.makedirs(image_dir, exist_ok=True)
    
    try:
        r = requests.get(product_link, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print(f"Connection error for {product_link}: {e}")
        return row
    
    title_element = soup.find("h1", class_="sf-heading__title title")
    if title_element:
        raw_title = title_element.get_text(strip=True)
        row["title"] = raw_title.strip()
    else:
        row["title"] = ""

    names = soup.find_all("dt", class_="sf-property__name")
    values = soup.find_all("dd", class_="sf-property__value")

    for name, value in zip(names, values):
        key = name.get_text(strip=True).lower().replace(":", "").replace(" ", "_")
        row[key] = value.get_text(strip=True)
    row["merged_info"] = " ".join(p.get_text(strip=True) for p in values)

    desc_container = soup.find("div", {"data-testid": "product-description"})
    if desc_container:
        row["description"] = desc_container.get_text(" ", strip=True)
    else:
        row["description"] = ""

    # Image Extraction and Download
    img_container = soup.find("span", class_="sf-image--wrapper sf-gallery__big-image")
    img_tag = img_container.find("img") if img_container else None
    
    img_url = None
    if img_tag:
        img_url = img_tag.get("src")
    
    row['image_url'] = img_url

    if img_url:
        safe_filename = re.sub(r'[^\w\s-]', '', row["title"]).strip().replace(' ', '_')
        save_path = os.path.join(image_dir, f"{safe_filename}.jpg")

        try:
            img_res = requests.get(img_url, headers=headers, stream=True, timeout=10)
            if img_res.status_code == 200 and 'image' in img_res.headers.get('Content-Type', ''):
                with open(save_path, 'wb') as f:
                    for chunk in img_res.iter_content(1024):
                        f.write(chunk)
                row["local_path"] = save_path
            else:
                print(f"Invalid image response for: {row['title']}")
        except Exception as e:
            print(f"Error downloading image for {row['title']}: {e}")
    return row


In [21]:
base_url = "https://www.lovecrafts.com/en-us/l/crochet/crochet-patterns/free-crochet-patterns?srsltid=AfmBOoryzbtxePMhSWrMgWx0Vuk-dkDGZwGOhSjdeTdpO3F9vnzuE115"
rows = []
page = 1

while True:
    print(f"Scraping page {page}...")
    url = base_url if page == 1 else f"{base_url}&p={page}"

    if page == 2:
        break

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    cards = soup.find_all("div", {"data-testid": "product-card"})
    
    if not cards:
        print("No more products found. Ending scrape.")
        break

    for card in cards:
        link_tag = card.find("a", class_="lc-product-card__link")
        
        if link_tag:
            href = link_tag.get("href")
            row = scrape_product(href, image_dir=image_dir)
            row["product_link"] = href
            
            rows.append(row)
            time.sleep(0.5)

    page += 1


df = pd.DataFrame(rows)

df.columns = [col.replace(':', '') for col in df.columns]
print("Scraping complete!")

Scraping page 1...
Scraping page 2...
Scraping complete!
